# Демо: OCR документов + финализация полей локальной HF-моделью

Ноутбук повторяет финальный подход: EasyOCR multi-pass + локальная HuggingFace-модель для финализации полей.


In [ ]:
!pip -q install easyocr opencv-python-headless matplotlib transformers sentencepiece

In [ ]:
import re
import cv2
import json
import easyocr
import matplotlib.pyplot as plt
from transformers import pipeline
from google.colab import files

In [ ]:
USE_GPU = False
OCR_CONF_THRESHOLD = 0.20
OCR_MIN_BOX_AREA = 60
HF_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

reader = easyocr.Reader(['ru', 'en'], gpu=USE_GPU)
hf_pipe = pipeline('text-generation', model=HF_MODEL)


def normalize_text(s):
    return re.sub(r'\s+', ' ', str(s).strip())


def is_text_like(s):
    return sum(ch.isalnum() for ch in s.strip()) >= 2


def build_ocr_variants(image_bgr):
    h, w = image_bgr.shape[:2]
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    sharp = cv2.filter2D(image_bgr, -1, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))
    up = cv2.resize(image_bgr, (int(w * 1.5), int(h * 1.5)), interpolation=cv2.INTER_CUBIC)
    return [('orig', image_bgr, 1.0), ('gray', gray, 1.0), ('clahe', clahe, 1.0), ('sharp', sharp, 1.0), ('up', up, 1.5)]


def bbox_from_quad(box):
    xs = [int(p[0]) for p in box]
    ys = [int(p[1]) for p in box]
    x, y = min(xs), min(ys)
    return x, y, max(xs)-x, max(ys)-y


def boxes_close(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    acx, acy = ax + aw/2, ay + ah/2
    bcx, bcy = bx + bw/2, by + bh/2
    return abs(acx-bcx) < 0.6*max(aw,bw,1) and abs(acy-bcy) < 0.6*max(ah,bh,1)


def merge_candidates(candidates):
    clusters = []
    for c in candidates:
        placed = False
        for cl in clusters:
            if boxes_close(c['bbox'], cl[0]['bbox']):
                cl.append(c)
                placed = True
                break
        if not placed:
            clusters.append([c])

    merged = []
    for cl in clusters:
        best = sorted(cl, key=lambda x: (x['confidence'], len(x['text'])), reverse=True)[0]
        x1 = min(x['bbox'][0] for x in cl)
        y1 = min(x['bbox'][1] for x in cl)
        x2 = max(x['bbox'][0] + x['bbox'][2] for x in cl)
        y2 = max(x['bbox'][1] + x['bbox'][3] for x in cl)
        merged.append({'text': best['text'], 'bbox': [x1, y1, x2-x1, y2-y1], 'confidence': round(sum(x['confidence'] for x in cl)/len(cl), 3)})
    return merged


def merge_words_into_lines(words):
    if not words:
        return []
    words = sorted(words, key=lambda x: (x['bbox'][1], x['bbox'][0]))
    heights = [max(1, w['bbox'][3]) for w in words]
    y_thr = max(8, int(0.6 * sorted(heights)[len(heights)//2]))

    clusters = []
    for w in words:
        y_mid = w['bbox'][1] + w['bbox'][3]/2
        placed = False
        for cl in clusters:
            cy = sum(x['bbox'][1] + x['bbox'][3]/2 for x in cl) / len(cl)
            if abs(y_mid - cy) <= y_thr:
                cl.append(w)
                placed = True
                break
        if not placed:
            clusters.append([w])

    lines = []
    for cl in clusters:
        cl = sorted(cl, key=lambda x: x['bbox'][0])
        text = normalize_text(' '.join(x['text'] for x in cl))
        if not is_text_like(text):
            continue
        x1 = min(x['bbox'][0] for x in cl)
        y1 = min(x['bbox'][1] for x in cl)
        x2 = max(x['bbox'][0] + x['bbox'][2] for x in cl)
        y2 = max(x['bbox'][1] + x['bbox'][3] for x in cl)
        conf = sum(x['confidence'] for x in cl)/len(cl)
        lines.append({'text': text, 'bbox': [x1, y1, x2-x1, y2-y1], 'confidence': round(conf, 3)})

    return sorted(lines, key=lambda x: (x['bbox'][1], x['bbox'][0]))


def run_ocr_multipass(image_bgr):
    candidates = []
    for _, variant, scale in build_ocr_variants(image_bgr):
        for box, text, conf in reader.readtext(variant, detail=1, paragraph=False):
            text = normalize_text(text)
            conf = float(conf)
            if conf < OCR_CONF_THRESHOLD or not is_text_like(text):
                continue
            x, y, w, h = bbox_from_quad(box)
            if scale != 1.0:
                x, y, w, h = int(x/scale), int(y/scale), int(w/scale), int(h/scale)
            if w*h < OCR_MIN_BOX_AREA:
                continue
            candidates.append({'text': text, 'bbox': [x,y,w,h], 'confidence': conf})
    return merge_words_into_lines(merge_candidates(candidates))


def heuristic_extract(lines):
    texts = [normalize_text(x['text']).upper() for x in lines]
    joined = ' '.join(texts)

    m = re.search(r'\b(\d{2}[./-]\d{2}[./-]\d{4})\b', joined)
    birth_date = m.group(1).replace('-', '.').replace('/', '.') if m else None

    m = re.search(r'\b(\d{2}\s?\d{2}\s?\d{6}|\d{9,12})\b', joined)
    doc_num = re.sub(r'\s+', ' ', m.group(1)).strip() if m else None

    stop = {'ВОДИТЕЛЬСКОЕ','УДОСТОВЕРЕНИЕ','ПАСПОРТ','РОССИЙСКАЯ','ФЕДЕРАЦИЯ','МВД'}
    full_name = None
    for t in texts:
        clean = re.sub(r'[^А-ЯЁ\s-]', '', t).strip()
        words = [w for w in clean.split() if w]
        if 2 <= len(words) <= 4 and not any(w in stop for w in words):
            if all(re.fullmatch(r'[А-ЯЁ-]{2,}', w) for w in words):
                full_name = ' '.join(words)
                break

    return {'full_name': full_name, 'birth_date': birth_date, 'document_number': doc_num}


def hf_extract(lines):
    line_texts = [x['text'] for x in lines]
    full_text = '\n'.join(line_texts)
    prompt = (
        'Извлеки поля full_name, birth_date, document_number из OCR. '
        'Не используй заголовки документа как ФИО. Верни только JSON.\n'
        f'OCR lines: {json.dumps(line_texts, ensure_ascii=False)}\n'
        f'OCR full_text: {full_text}'
    )
    out = hf_pipe(prompt, max_new_tokens=160, do_sample=False, return_full_text=False)
    txt = out[0]['generated_text']
    m = re.search(r'\{[\s\S]*\}', txt)
    if not m:
        return None, f'HF model returned non-JSON: {txt[:200]}'
    obj = json.loads(m.group(0))
    return {
        'full_name': obj.get('full_name'),
        'birth_date': obj.get('birth_date'),
        'document_number': obj.get('document_number')
    }, None


def draw_lines(image_bgr, lines):
    out = image_bgr.copy()
    for l in lines:
        x, y, w, h = l['bbox']
        cv2.rectangle(out, (x, y), (x+w, y+h), (0, 255, 0), 2)
    return out

In [ ]:
uploaded = files.upload()
image_name = next(iter(uploaded.keys()))
image_bgr = cv2.imread(image_name)

lines = run_ocr_multipass(image_bgr)
heur = heuristic_extract(lines)
hf_fields, hf_error = hf_extract(lines)
final_fields = hf_fields if hf_fields else heur

print('OCR lines:', len(lines))
print('Heuristic fields:')
print(json.dumps(heur, ensure_ascii=False, indent=2))
print('HF fields:')
print(json.dumps(hf_fields, ensure_ascii=False, indent=2) if hf_fields else hf_error)
print('Final fields:')
print(json.dumps(final_fields, ensure_ascii=False, indent=2))

annotated = draw_lines(image_bgr, lines)
plt.figure(figsize=(9, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Detected OCR lines')
plt.show()

In [ ]:
result = {
    'ocr': {
        'engine': 'easyocr_multipass',
        'lines': lines,
        'full_text': '\n'.join([x['text'] for x in lines]),
        'lines_count': len(lines),
    },
    'fields': final_fields,
    'heuristic_fields': heur,
    'hf_fields': hf_fields,
    'hf_error': hf_error,
}

with open('result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print('Saved: result.json')
files.download('result.json')